In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import xarray as xr
from scipy.io import savemat
from hydrobm.calculate import calc_bm
import hydrobm.benchmarks as _bm_module

# All 22 available benchmarks
ALL_BENCHMARKS = [name[3:] for name in dir(_bm_module) if name.startswith("bm_")]
print(f"{len(ALL_BENCHMARKS)} benchmarks:", ALL_BENCHMARKS)

# Load a single Caravan-2 catchment CSV and prepare it for hydrobm

In [ ]:
import xarray as xr

SUBSET_DIR = Path("<LOCAL_DATA>/caravan_subset")

def load_catchment(nc_path: Path) -> pd.DataFrame:
    """Load a Caravan subset NetCDF and return a DataFrame ready for hydrobm."""
    ds = xr.open_dataset(nc_path)
    df = ds[["total_precipitation_sum", "streamflow", "temperature_2m_mean"]].to_dataframe()
    df.index.name = "date"
    df = df.rename(columns={
        "total_precipitation_sum": "precipitation",
        "temperature_2m_mean": "temperature",
    })
    return df

# Preview one file
example = next(SUBSET_DIR.glob("*.nc"))
df_ex = load_catchment(example)
print(f"{example.stem}: {len(df_ex)} days")
df_ex[["precipitation", "streamflow"]].describe()

In [ ]:
def get_cal_period(gauge_id: str):
    """
    Calibration period per bm_plots_new_runs.m:
      - camelsaus_607155: 1990-01-01 – 1999-12-31
      - all others:       2005-01-01 – 2014-12-31
    """
    if gauge_id == "camelsaus_607155":
        return "1990-01-01", "1999-12-31"
    return "2005-01-01", "2014-12-31"

In [ ]:
OUTPUT_DIR = Path("<LOCAL_ROOT>/benchmark_results")
OUTPUT_DIR.mkdir(exist_ok=True)

def to_matlab_datenum(index: pd.DatetimeIndex) -> np.ndarray:
    return np.array([ts.toordinal() + 366 for ts in index], dtype=np.float64)

for nc_path in sorted(SUBSET_DIR.glob("*.nc")):
    gauge_id = nc_path.stem
    df = load_catchment(nc_path)

    cal_start, cal_end = get_cal_period(gauge_id)
    df_cal = df.loc[cal_start:cal_end]
    n = len(df_cal)
    cal_mask = np.ones(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)  # workaround: hydrobm requires same-length mask

    benchmark_flows, scores = calc_bm(
        data=df_cal,
        cal_mask=cal_mask,
        val_mask=val_mask,
        precipitation="precipitation",
        streamflow="streamflow",
        benchmarks=ALL_BENCHMARKS,
        metrics=["kge", "nse", "rmse"],
    )

    mat_data = {
        "gauge_id":        gauge_id,
        "dates":           to_matlab_datenum(benchmark_flows.index),
        "benchmark_flows": benchmark_flows.values.astype(np.float64),         # [T x N_benchmarks]
        "benchmark_names": np.array(benchmark_flows.columns.tolist(), dtype=object),
        "q_obs":           df_cal["streamflow"].values.astype(np.float64),    # [T x 1] observed flow
        "precip_obs":      df_cal["precipitation"].values.astype(np.float64), # [T x 1] precipitation
        "cal_start":       cal_start,
        "cal_end":         cal_end,
    }
    savemat(OUTPUT_DIR / f"{gauge_id}_benchmarks.mat", mat_data)
    print(f"Saved: {gauge_id}  ({n} days, {cal_start} – {cal_end})")

print(f"\nDone. Files in: {OUTPUT_DIR}")